In [109]:
# We use ir for Creating Tensors, Mathematical operations, automatic differentiation, Training Neural Networks
import torch

# nn contains the tools for building neural networks
from torch import nn

# TensorDataset --> This helps us combine input data and target into a dataset. It simply organizes our data, it doesn't train the model
# DataLoader --> This helps us feed the dataset to our model in smaller groups called batches
# random_split --> This divides our dataset into separate portions, usually training and validation data
from torch.utils.data import TensorDataset, DataLoader, random_split

# A simple Regression problem: y = 3x + 2

In [110]:
# Controls Randomness in Pytorch
torch.manual_seed(42)

## Step 1: Prepare the data

In [111]:
# Generate input features - 100 numbers that are evenly spaced between -1 and 1
X = torch.linspace(-1, 1, 100).reshape(-1, 1)

In [112]:
# Generate target values with a little noise
y = 3*X + 2 + 0.1 * torch.randn_like(X)

In [113]:
# Combine features and targets into a dataset using TensorDataset
dataset = TensorDataset(X, y)

In [114]:
dataset[2]  # We get the third input-target pair

(tensor([-0.9596]), tensor([-0.7887]))

### Split the dataset into training and validation

In [115]:
train_size = 80     # 80 samples for the training
val_size = 20       # 20 samples for validation

In [116]:
# random_split selects samples randomly, but it doesn't change the original X and y tensors. Also, random_split returns two Subset objects. Each one refers to selected indices in the original dataset
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

print(len(train_dataset))
print(len(val_dataset))

80
20


### Understanding DataLoader

In [117]:
# DataLoader sends a small number of samples (called batches) into the model, instead of all the data at once

# batch_size = 16 --> We have 80 samples, so 80/16 = 5 batches
# batch 1: 16 samples, batch 2: 16 samples, batch 3: 16 samples ...
# The model processes one batch, calculates the loss, computes the gradients, and updates its parameters.
# Then it processes the next batch. This continues until all five batches have been processed
# Shuffle = true --> It randomly changes the order of training samples when a new iteration over the DataLoader begins. Shuffling can help avoid learning patterns that are caused only by the ordering of the training examples. Shuffle should be False for validation/testing
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

print(len(train_loader))
print(len(val_loader))

5
2


### Inspect the actual data ourselves

In [118]:
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print("-" * 20)

print(f"Total Samples: {len(dataset)}")
print("-" * 20)

print(f"Training Samples: {len(train_dataset)}")
print(f"Validation Samples: {len(val_dataset)}")
print("-" * 20)

print(f"Training Batches: {len(train_loader)}")
print(f"Validation Batches: {len(val_loader)}")

X shape: torch.Size([100, 1])
y shape: torch.Size([100, 1])
--------------------
Total Samples: 100
--------------------
Training Samples: 80
Validation Samples: 20
--------------------
Training Batches: 5
Validation Batches: 2


In [119]:
# Inspect one training batch
X_batch, y_batch = next(iter(train_loader))

In [120]:
print("Batch X shape:", X_batch.shape)
print("Batch y shape:", y_batch.shape)

print("First 5 inputs:")
print(X_batch[:5])

print("First 5 targets:")
print(y_batch[:5])

Batch X shape: torch.Size([16, 1])
Batch y shape: torch.Size([16, 1])
First 5 inputs:
tensor([[ 0.3939],
        [ 0.3131],
        [-0.6364],
        [-0.8586],
        [ 0.9394]])
First 5 targets:
tensor([[ 3.1621],
        [ 3.0250],
        [ 0.0412],
        [-0.7362],
        [ 4.7195]])


## Step 2: Define the model (neural network)

In [121]:
# In pytorch, nn.Module is the base class for building Neural Networks. It provides functionality for:
# - Defining layers.
# - Managing learnable parameters.
# - Organizing multiple layers.
# - Switching between training and evaluation modes.

In [122]:
# This means our model receives ine input feature (X), and produces one output value (y)
# Pytorch automatically creates two learnable parameters (weight (w), bias(b))
model = nn.Linear(in_features=1, out_features=1)

print(model.weight)
print(model.bias)

Parameter containing:
tensor([[-0.3809]], requires_grad=True)
Parameter containing:
tensor([-0.6473], requires_grad=True)


In [123]:
# Inspect their shapes
print(model.weight.shape)   # One Input and One output feature
print(model.bias.shape)     # One output

torch.Size([1, 1])
torch.Size([1])


## Step 3: Loss Function and Optimizer

In [125]:
# Actual Values
y_actual = torch.tensor([[10.0], [20.0], [30.0]])
# Predicted Values
y_pred = torch.tensor([[8.0], [18.0], [25.0]])

# Create the loss function
loss_fn = nn.MSELoss()

# Calculate the loss
loss = loss_fn(y_pred, y_actual)

print(f"Loss: {loss.item()}")

Loss: 11.0


In [126]:
# Create an MSE loss function object - It doesn't calculate any loss yet
loss_function = nn.MSELoss()

In [ ]:
# Creating an optimizer

# model.parameters() --> It gives the optimizer access to the model's learnable parameters. For our model, they are model.weight, and model.bias
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

## Step 4: The training Loop

In [ ]:
def train_one_epoch(model, dataloader, loss_function, optimizer):

    # this tells pytorch that we are using the model in training mode
    model.train()

    total_loss = 0.0

    # X_batch is the 16 input samples
    # y_batch is the corresponding 16 actual targets
    for X_batch, y_batch in dataloader:
        # 1 - Clear old gradients from the previous batch because PyTorch accumulates gradients by default.
        optimizer.zero_grad()

        # 2 - Forward Pass - The model takes the inputs and generates predictions.
        predictions = model(X_batch)

        # 3 - Calculate the error
        loss = loss_function(predictions, y_batch)

        # 4 - Backpropagation: calculate gradients. Uses autograd to calculate gradients for the learnable parameters. It does not update those parameters itself.
        loss.backward()

        # 5 - Update model parameters
        optimizer.step()

        # Track loss for the whole epoch
        total_loss =  total_loss + loss.item() * X_batch.size(0)

    return total_loss / len(dataloader.dataset)